In [ ]:
import numpy as np
import pandas as pd
import numpy as np
import nltk
import matplotlib
import seaborn
import textblob
import plotly
import cufflinks
import imblearn
print("All main packages imported successfully!")

#NLTK libraries
import nltk
import re
import string
from wordcloud import WordCloud,STOPWORDS
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn import svm, datasets
from sklearn import preprocessing

from sklearn import metrics
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve, auc

import matplotlib.pyplot as plt
from matplotlib import rcParams
import seaborn as sns
from textblob import TextBlob
from plotly import tools
import plotly.graph_objs as go
from plotly.offline import iplot
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')


from scipy.interpolate import interp1d
from itertools import cycle
import cufflinks as cf
from collections import defaultdict
from collections import Counter
from imblearn.over_sampling import SMOTE

In [ ]:
df = pd.read_csv('sentiment-analysis-project/convertcsv.csv')

In [ ]:
df.head()

In [ ]:
df.tail(10)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df['reviewText']=df['reviewText'].fillna('Missing')

df.isnull().sum()

In [ ]:
df['reviews']=df['reviewText']+df['summary']
df=df.drop(['reviewText', 'summary'], axis=1)
df.head()

In [ ]:
if 'helpful/0' in df.columns and 'helpful/1' in df.columns:
    df['helpful'] = df[['helpful/0' ,  'helpful/1']].values.tolist()
    df = df.drop(['helpful/0', 'helpful/1'], axis=1)
display(df.head())


In [ ]:
df['overall'].value_counts()

In [ ]:
def f(row):

    #This function returns sentiment value based on the overall ratings from the user

    if row['overall'] == 3.0:
        val = 'Neutral'
    elif row['overall'] == 1.0 or row['overall'] == 2.0:
        val = 'Negative'
    elif row['overall'] == 4.0 or row['overall'] == 5.0:
        val = 'Positive'
    else:
        val = -1
    return val

In [ ]:
df['sentiment'] = df.apply(f,axis=1)
df.head()

In [ ]:
df['sentiment'].value_counts()

In [ ]:
# new data frame which has date and year
new = df["reviewTime"].str.split(",", n = 1, expand = True)

# making separate date column from new data frame
df["date"]= new[0]

# making separate year column from new data frame
df["year"]= new[1]

df=df.drop(['reviewTime'], axis=1)
df.head()

In [ ]:
# Splitting the date
new1 = df["date"].str.split(" ", n = 1, expand = True)

# adding month to the main dataset
df["month"]= new1[0]

# adding day to the main dataset
df["day"]= new1[1]

df=df.drop(['date'], axis=1)
df.head()

In [ ]:
#Creating a dataframe
day=pd.DataFrame(df.groupby('day')['reviews'].count()).reset_index()
day['day']=day['day'].astype('int64')
day.sort_values(by=['day'])

#Plotting the graph
sns.barplot(x="day", y="reviews", data=day)
plt.title('Day vs Reviews count')
plt.xlabel('Day')
plt.ylabel('Reviews count')

In [ ]:
# Calculate helpfulness rate from the list [helpful/0, helpful/1]
def calculate_helpful_rate(helpful_list):
    if helpful_list[0] == 0:
        return 0
    else:
        return round(helpful_list[0] / helpful_list[1], 2)

df['helpful_rate'] = df['helpful'].apply(calculate_helpful_rate)

# dropping the helpful column from main dataframe
# Check if 'helpful' column exists before dropping
if 'helpful' in df.columns:
    df = df.drop(['helpful'], axis=1)

display(df.head())

In [ ]:
df['helpful_rate'].value_counts()

In [ ]:
df=df.drop(['reviewerName','unixReviewTime'], axis=1)

clean_reviews=df.copy()

In [ ]:
sns.countplot(x='year', hue='sentiment', data=df)
plt.show()

In [ ]:
df.groupby(['year','sentiment'])['sentiment'].count().unstack().plot(legend=True)
plt.title('Year and Sentiment count')
plt.xlabel('Year')
plt.ylabel('Sentiment count')
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Plotting the 'overall' ratings histogram
sns.histplot(df['overall'], bins=5, kde=False, color='skyblue', edgecolor='black')

# Adding plot details
plt.title('Review Rating Distribution')
plt.xticks([1, 2, 3, 4, 5])  # Set the x-axis ticks to only show integer values
plt.xlabel('Rating')
plt.ylabel('Count')

# Display the plot
plt.show()

In [ ]:
import re
def review_cleaning(text):
    text = str(text).lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    return text

In [ ]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    return " ".join([word for word in str(text).split() if word not in stop_words])

df['clean_text'] = df['reviews'].apply(review_cleaning)
df['clean_text'] = df['clean_text'].apply(remove_stopwords)

In [ ]:
df['reviews'] = df['reviews'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stop_words)]))
df.head()

In [ ]:
df = df.drop('reviews', axis=1)
df.head()

Stemming the reviews

Stemming is a method of deriving root word from the inflected word. Here we extract the clean_text and convert the words in reviews to its root word. for example,

Going->go
Finally->fina


If you notice, the root words doesn't need to carry a semantic meaning. There is another technique knows as Lemmatization where it converts the words into root words which has a semantic meaning. Simce it takes time.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

# calling the label encoder function
label_encoder = preprocessing.LabelEncoder()

# Encode labels in column 'sentiment'.
df['sentiment']= label_encoder.fit_transform(df['sentiment'])

df['sentiment'].unique()

In [ ]:
review_features=df.copy()
review_features = review_features[['clean_text']].reset_index(drop=True)
review_features.head()

In [ ]:
from nltk.stem.porter import PorterStemmer

#Performing stemming on the review dataframe
ps = PorterStemmer()

#splitting and adding the stemmed words except stopwords
corpus = []
for i in range(0, len(review_features)):
    review = re.sub('[^a-zA-Z]', ' ', review_features['clean_text'][i])
    review = review.split()
    review = [ps.stem(word) for word in review if not word in stop_words]
    review = ' '.join(review)
    corpus.append(review)

In [ ]:
corpus[9]


TFIDF(Term Frequency — Inverse Document Frequency)


TF-IDF stands for “Term Frequency — Inverse Document Frequency”. This is a technique to quantify a word in documents, we generally compute a weight to each word which signifies the importance of the word in the document and corpus. This method is a widely used technique in Information Retrieval and Text Mining.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_features=5000,ngram_range=(1,2))
# TF-IDF feature matrix
X= tfidf_vectorizer.fit_transform(review_features['clean_text'])
X.shape

In [ ]:
y=df['sentiment']

Handling Imbalance target feature-SMOTE


In our target feature, we noticed that we got a lot of positive sentiments compared to negative and neutral. So it is crucial to balanced the classes in such situations. Here I have used SMOTE(Synthetic Minority Oversampling Technique) to balance out the imbalanced dataset problem.It aims to balance class distribution by randomly increasing minority class examples by replicating them.

SMOTE synthesises new minority instances between existing minority instances. It generates the virtual training records by linear interpolation for the minority class. These synthetic training records are generated by randomly selecting one or more of the k-nearest neighbors for each example in the minority class. After the oversampling process, the data is reconstructed and several classification models can be applied for the processed data.


In [ ]:
# from collections import Counter
# from imblearn.over_sampling import SMOTE

# print(f'Original dataset shape : {Counter(y)}')

# smote = SMOTE(random_state=42)
# X_res, y_res = smote.fit_resample(X, y)

# print(f'Resampled dataset shape {Counter(y_res)}')

In [ ]:
from imblearn.over_sampling import RandomOverSampler
from collections import Counter

print(f'Original dataset shape : {Counter(y)}')

ros = RandomOverSampler(random_state=42)
X_res_ros, y_res_ros = ros.fit_resample(X, y)

print(f'Resampled dataset shape using RandomOverSampler: {Counter(y_res_ros)}')

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_res_ros, y_res_ros, test_size=0.25, random_state=42)

In [ ]:
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# model = RandomForestClassifier(n_estimators=100, random_state=42)
# model.fit(X_train,y_train)

In [ ]:
# y_pred = model.predict(X_test)
# print("Accuracy:", accuracy_score(y_test, y_pred))
# print(classification_report(y_test,y_pred))

In [ ]:
# for n in [10, 50, 100, 200]:
#     clf = RandomForestClassifier(n_estimators=n, random_state=42)
#     clf.fit(X_train, y_train)
#     acc = accuracy_score(y_test, clf.predict(X_test))
#     print(f"n_estimators = {n} → Accuracy = {acc:.4f}")

In [ ]:
# from sklearn.pipeline import Pipeline
# from sklearn.linear_model import LogisticRegression

# # Create a pipeline that trains the model
# model = Pipeline([
#     ('logistic', LogisticRegression())
# ])

# model.fit(X_train, y_train)

In [ ]:
# from sklearn.svm import SVC
# from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# # Define class weights
# weights = {0: 0.2, 1: 0.3, 2: 0.5}

# svm_model = SVC(kernel='rbf', probability=True, class_weight=weights)
# svm_model.fit(X_train, y_train)
# y_pred = svm_model.predict(X_test)
# accuracy_svm = accuracy_score(y_test, y_pred)
# conf_matrix_svm = confusion_matrix(y_test, y_pred)
# class_report_svm = classification_report(y_test, y_pred)

# print(f"SVM Model Accuracy: {accuracy_svm:.2f}")
# print("\nSVM Model Confusion Matrix:")
# print(conf_matrix_svm)
# print("\nSVM Model Classification Report:")
# print(class_report_svm)

In [ ]:
from sklearn.linear_model import LogisticRegression

weights = {0:1 , 1:1.2 , 2:1.4} # Note: Finding optimal weights often requires hyperparameter tuning.

model = LogisticRegression(class_weight=weights, multi_class="multinomial", solver="lbfgs", max_iter=2000)
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy:.2f}")
print("\nConfusion Matrix:")
print(conf_matrix)
print("\nClassification Report:")
print(class_report)

In [ ]:
# from sklearn.neighbors import KNeighborsClassifier

In [ ]:
# model = KNeighborsClassifier(n_neighbors=3)
# model.fit(X_train, y_train)

# y_pred = model.predict(X_test)
# from sklearn.metrics import accuracy_score
# accuracy = accuracy_score(y_test, y_pred)
# print("accuracy: ", accuracy)


In [ ]:
# from sklearn.model_selection import cross_val_score
# from sklearn.neighbors import KNeighborsClassifier

# # Initialize the KNN model
# knn_model_original = KNeighborsClassifier(n_neighbors=3)

# # Perform 10-fold cross-validation on the original data (X, y)
# cv_scores_knn_original = cross_val_score(knn_model_original, X, y, cv=10)

# # Print the cross-validation scores and the mean accuracy
# print("Cross-validation scores for KNN on original data:", cv_scores_knn_original)

**Logistic Regression with Hyperparameter tuning**


We use regularization parameter and penality for parameter tuning. let's see which one to plug.

In [ ]:
# param_grid = {'C': np.logspace(-4, 4, 50),
#              'penalty':['l1', 'l2']}
# clf = GridSearchCV(LogisticRegression(random_state=0), param_grid,cv=5, verbose=0,n_jobs=-1)
# best_model = clf.fit(X_train,y_train)
# print(best_model.best_estimator_)
# print("The mean accuracy of the model is:",best_model.score(X_test,y_test))

In [ ]:
# laogreg = LogisticRegression(C=10000.0, random_state=42)
# # Correcting curly quotes to straight quotes
# Multi_class = "multinomial"
# logreg.fit(X_test, y_test)
# y_pred = logreg.predict(X_test)
# print('Accuracy of logistic regression classifier on test set: {:.2f}'.format(logreg.score(X_test, y_test)))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns


cm_original = [[2185,   17,   23],
 [   1, 2266,   10],
 [  27,  264, 1974]]


plt.figure(figsize=(8, 6))
sns.heatmap(cm_original, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Neutral', 'Positive'], yticklabels=['Negative', 'Neutral', 'Positive'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix (Model trained on Original data)')
plt.show()

In [ ]:
print("Classification Report:\n",classification_report(y_test, y_pred))

In [ ]:
import joblib

# Save the model and TF-IDF vectorizer
joblib.dump(model,'sentiment_model.pkl')  # Save the trained model (Logistic Regression)
joblib.dump(tfidf_vectorizer, 'tfidf_vectorizer.pkl')  # Save the TF-IDF vectorizer

In [ ]:
# Load the model and vectorizer
model = joblib.load('sentiment_model.pkl')
vectorizer = joblib.load('tfidf_vectorizer.pkl')

In [ ]:
import re
import string

def review_cleaning(text):

    text = str(text).lower()  # Convert to lowercase
    text = re.sub('\[.*?\]', '', text)  # Remove text in square brackets
    text = re.sub('https?://\S+|www\.\S+', '', text)  # Remove URLs
    text = re.sub('<.*?>+', '', text)  # Remove HTML tags
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)  # Remove punctuation
    text = re.sub('\n', '', text)  # Remove newlines
    text = re.sub('\w*\d\w*', '', text)  # Remove words with numbers
    return text

In [ ]:
from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    return " ".join([word for word in str(text).split() if word not in stop_words])
    return ' '.join([word for word in text.split() if word not in stop_words])

In [ ]:
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

def perform_stemming(text):
    """
    Apply stemming to a given text using PorterStemmer.
    """
    words = text.split()
    stemmed_words = [ps.stem(word) for word in words]
    return ' '.join(stemmed_words)

In [ ]:
def preprocess_text(text):
    """
    Full preprocessing pipeline: cleaning, removing stopwords, and stemming.
    """
    text = review_cleaning(text)  # Clean the text
    text = remove_stopwords(text)  # Remove stopwords
    text = perform_stemming(text)  # Apply stemming
    return text

In [ ]:
# Example input text
text = "The product quality is good."

# Preprocess and vectorize the input
preprocessed_text = preprocess_text(text)
input_vectorized = vectorizer.transform([preprocessed_text])  # Convert to TF-IDF features

In [ ]:
# Predict sentiment
prediction = model.predict(input_vectorized)
sentiment_classes = ['Negative', 'Neutral', 'Positive']
predicted_sentiment = sentiment_classes[prediction[0]]

print(f"Predicted Sentiment: {predicted_sentiment}")

In [ ]:
# Example input text
text = "The product quality is very bad ."

# Preprocess and vectorize the input
preprocessed_text = preprocess_text(text)
input_vectorized = tfidf_vectorizer.transform([preprocessed_text])   # Use the updated tfidf_vectorizer

In [ ]:
# Predict sentiment
prediction = model.predict(input_vectorized)
sentiment_classes = ['Negative', 'Neutral', 'Positive']
predicted_sentiment = sentiment_classes[prediction[0]]

print(f"Predicted Sentiment: {predicted_sentiment}")

In [ ]:
# A list of 20 diverse text examples
example_texts_20 = [
    "This product is absolutely fantastic! Highly recommended.", # Positive
    "It's okay, nothing special. Works as advertised.", # Neutral
    "Very disappointed with the quality. It broke after a week.", # Negative
    "The battery life is excellent, lasts all day.", # Positive
    "The instructions were a bit confusing.", # Neutral/Negative
    "Amazing value for the price, couldn't be happier.", # Positive
    "It's just average, not great, not terrible.", # Neutral
    "Customer service was unhelpful and rude.", # Negative
    "Works perfectly fine for my needs.", # Positive
    "The color is slightly different from the picture.", # Neutral
    "This is the worst product I've ever bought.", # Negative
    "Easy to set up and use.", # Positive
    "The packaging was damaged upon arrival.", # Negative
    "Good quality materials used.", # Positive
    "It's a bit smaller than I expected.", # Neutral
    "I would not recommend this product.", # Negative
    "Very durable and sturdy.", # Positive
    "The sound quality is decent.", # Neutral
    "Experienced some issues with the software.", # Negative
    "Overall, a satisfactory purchase." # Positive/Neutral
]

# Preprocess and predict sentiment for each example
print("Testing with 20 diverse text examples:")
for i, text in enumerate(example_texts_20):
    preprocessed_text = preprocess_text(text)
    input_vectorized = tfidf_vectorizer.transform([preprocessed_text])

    # Predict sentiment
    prediction = model.predict(input_vectorized)
    predicted_sentiment = sentiment_classes[prediction[0]]

    # Predict probabilities
    probabilities = model.predict_proba(input_vectorized)

    print(f"\nExample {i+1}: '{text}'")
    print(f"Predicted Sentiment: {predicted_sentiment}")
    print("Probabilities:")
    for j, sentiment in enumerate(sentiment_classes):
        print(f"  {sentiment}: {probabilities[0][j]:.2f}")